# Il suono — sei demo interattive

Le demo riprendono i concetti del capitolo 1 della dispensa di **Informazione Multimediale · Audio**.

1. Eseguire la cella **Preparazione** selezionandola e premendo **Shift + Invio** (oppure ▶).
2. Eseguire la demo scelta; modificare i parametri iniziali e rieseguirla per confrontare i risultati.
3. Per l’ascolto, avviare il lettore audio mostrato sotto la cella. Iniziare con un volume moderato.

La prima esecuzione carica Python e le librerie: può richiedere qualche istante. I calcoli avvengono nel browser. Le modifiche restano in questo browser; per conservarle come file usare **File → Save and Export Notebook As → Notebook**.

Le formule e le operazioni sono esplicite: questa versione didattica usa NumPy, Matplotlib e IPython e non richiede `audiolab` né file sul computer.


## Preparazione

La frequenza di campionamento è 44.100 Hz. Gli ascolti non vengono normalizzati separatamente: differenze di ampiezza e cancellazioni restano udibili. Brevi rampe ai bordi evitano clic artificiali.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

FS = 44100
plt.rcParams.update({"figure.figsize": (9, 3), "axes.grid": True, "grid.alpha": 0.25})

def tempo(durata):
    assert 0 < durata <= 10, "Usare una durata fra 0 e 10 secondi."
    return np.arange(round(FS * durata)) / FS

def ascolto(y, etichetta):
    y = np.asarray(y, dtype=float).copy()
    assert np.max(np.abs(y)) <= 1, "Picco oltre fondo scala: ridurre le ampiezze."
    n = min(round(0.01 * FS), len(y) // 2)
    if n:
        rampa = np.linspace(0, 1, n)
        y[:n] *= rampa
        y[-n:] *= rampa[::-1]
    print(etichetta)
    display(Audio(y, rate=FS, normalize=False))

print("Python pronto. Le sei demo possono essere eseguite nell’ordine preferito.")


## 1 — Ampiezza, frequenza e fase

$x(t)=A\sin(2\pi ft+\varphi)$. Modificare un parametro alla volta: l’ampiezza cambia la scala verticale, la frequenza il numero di oscillazioni e la fase l’allineamento temporale. Il grafico mostra i primi 20 ms; l’ascolto dura un secondo.


In [ ]:
# Parametri da modificare
ampiezza = 0.2        # fra 0 e 0.8
frequenza = 440.0     # Hz, fra 20 e 5000
fase = 0.0           # radianti; provare np.pi / 2

assert 0 <= ampiezza <= 0.8
assert 20 <= frequenza <= 5000
t = tempo(1.0)
y = ampiezza * np.sin(2 * np.pi * frequenza * t + fase)
n = round(0.020 * FS)
plt.plot(t[:n] * 1000, y[:n], color="#1769c2")
plt.ylim(-1, 1)
plt.xlabel("Tempo [ms]"); plt.ylabel("Ampiezza digitale")
plt.title(f"Sinusoide: {frequenza:g} Hz — periodo {1000 / frequenza:.2f} ms")
plt.tight_layout(); plt.show(); plt.close()
ascolto(y, "Sinusoide")


## 2 — Interferenza controllata dalla fase

Due toni alla stessa frequenza possono rinforzarsi o cancellarsi. Con ampiezze uguali, la somma ha ampiezza $2A|\cos(\Delta\varphi/2)|$. Provare sfasamenti di 0, 90 e 180 gradi; a 180° il silenzio è il risultato previsto, non un errore del lettore.


In [ ]:
# Parametri da modificare
sfasamento_gradi = 180   # provare 0, 90, 180
frequenza = 440.0
ampiezza = 0.2

assert 20 <= frequenza <= 5000 and 0 <= ampiezza <= 0.4
t = tempo(1.0)
x1 = ampiezza * np.sin(2 * np.pi * frequenza * t)
x2 = ampiezza * np.sin(2 * np.pi * frequenza * t + np.deg2rad(sfasamento_gradi))
y = x1 + x2
n = round(0.015 * FS)
plt.plot(t[:n] * 1000, x1[:n], alpha=0.55, label="Tono 1")
plt.plot(t[:n] * 1000, x2[:n], alpha=0.55, label="Tono 2")
plt.plot(t[:n] * 1000, y[:n], color="#17365d", label="Somma", linewidth=2)
plt.ylim(-0.9, 0.9); plt.xlabel("Tempo [ms]"); plt.ylabel("Ampiezza")
plt.title(f"Interferenza: sfasamento {sfasamento_gradi:g}°"); plt.legend()
plt.tight_layout(); plt.show(); plt.close()
ascolto(y, "Somma dei due toni, senza normalizzazione")


## 3 — Vedere e ascoltare i battimenti

Due toni vicini producono pulsazioni a $f_b=|f_2-f_1|$. Modificare la differenza di frequenza e confrontare la velocità dei battimenti. Le curve tratteggiate delimitano l’inviluppo della somma, non sono ulteriori suoni.


In [ ]:
# Parametri da modificare
f1 = 440.0
differenza_hz = 3.0    # provare 1, 3, 6
ampiezza = 0.18

f2 = f1 + differenza_hz
assert 20 <= f1 <= 5000 and 20 <= f2 <= 5000 and 0 <= ampiezza <= 0.4
t = tempo(3.0)
y = ampiezza * (np.sin(2 * np.pi * f1 * t) + np.sin(2 * np.pi * f2 * t))
inviluppo = 2 * ampiezza * np.abs(np.cos(np.pi * differenza_hz * t))
plt.plot(t, y, color="#1769c2", linewidth=0.35)
plt.plot(t[::100], inviluppo[::100], "--", color="#d9362b")
plt.plot(t[::100], -inviluppo[::100], "--", color="#d9362b")
plt.xlabel("Tempo [s]"); plt.ylabel("Ampiezza")
plt.title(f"Battimenti: {abs(differenza_hz):g} pulsazioni al secondo")
plt.tight_layout(); plt.show(); plt.close()
ascolto(y, f"Toni a {f1:g} e {f2:g} Hz")


## 4 — Stessa fondamentale, timbro diverso

Un tono puro e una somma di armoniche condividono la fondamentale. Modificare il numero di armoniche per ascoltare il cambiamento del timbro. Le due versioni sono portate allo **stesso RMS prima delle rampe**, per limitare le differenze di livello fisico: questo non garantisce uguale sonorità percepita.


In [ ]:
# Parametri da modificare
fondamentale = 220.0
numero_armoniche = 6    # da 1 a 12
rms_comune = 0.12

assert isinstance(numero_armoniche, int) and 1 <= numero_armoniche <= 12
assert 20 <= fondamentale and numero_armoniche * fondamentale < FS / 2
assert 0 < rms_comune <= 0.15
t = tempo(1.5)
puro = np.sin(2 * np.pi * fondamentale * t)
complesso = sum(np.sin(2 * np.pi * k * fondamentale * t) / k
                for k in range(1, numero_armoniche + 1))
puro *= rms_comune / np.sqrt(np.mean(puro**2))
complesso *= rms_comune / np.sqrt(np.mean(complesso**2))
n = round(0.020 * FS)
plt.plot(t[:n] * 1000, puro[:n], label="Tono puro")
plt.plot(t[:n] * 1000, complesso[:n], label=f"{numero_armoniche} armoniche", alpha=0.8)
plt.ylim(-0.6, 0.6); plt.xlabel("Tempo [ms]"); plt.ylabel("Ampiezza")
plt.title("Forma d’onda e timbro"); plt.legend()
plt.tight_layout(); plt.show(); plt.close()
ascolto(puro, "Tono puro")
ascolto(complesso, "Somma armonica")


## 5 — Ampiezza e decibel

Per lo stesso segnale moltiplicato per un guadagno $g$, la variazione di livello è $\Delta L=20\log_{10}(g)$ dB. Modificare il guadagno: il confronto conserva un’unica scala di ascolto e non normalizza le due versioni. Si tratta di un rapporto digitale, non di una misura in dB SPL.


In [ ]:
# Parametri da modificare
variazione_db = -6.0     # da -24 a +12 dB
ampiezza_base = 0.1

assert -24 <= variazione_db <= 12 and 0 < ampiezza_base <= 0.1
t = tempo(1.0)
x = ampiezza_base * np.sin(2 * np.pi * 440 * t)
guadagno = 10 ** (variazione_db / 20)
y = guadagno * x
print(f"Guadagno d’ampiezza: {guadagno:.3f}; variazione: {variazione_db:g} dB")
n = round(0.015 * FS)
plt.plot(t[:n] * 1000, x[:n], label="Riferimento")
plt.plot(t[:n] * 1000, y[:n], label="Elaborato")
plt.ylim(-0.5, 0.5); plt.xlabel("Tempo [ms]"); plt.ylabel("Ampiezza")
plt.title("Un riferimento comune per il confronto"); plt.legend()
plt.tight_layout(); plt.show(); plt.close()
ascolto(x, "Riferimento")
ascolto(y, f"Versione con variazione di {variazione_db:g} dB")


## 6 — Distanza della parete ed eco

La riflessione percorre andata e ritorno: il ritardo è $\tau=2d/c$, con $c=343$ m/s. Modificare la distanza della parete e il peso della riflessione. Con distanze piccole il ritorno tende a fondersi con il suono diretto; con distanze maggiori si distingue come ripetizione. Il modello contiene una sola riflessione.


In [ ]:
# Parametri da modificare
distanza_parete = 50.0      # metri, da 1 a 100
peso_riflessione = 0.5      # da 0 a 0.8

assert 1 <= distanza_parete <= 100 and 0 <= peso_riflessione <= 0.8
t_ping = tempo(0.08)
ping = 0.4 * np.sin(2 * np.pi * 900 * t_ping) * np.exp(-55 * t_ping)
n_fade = round(0.003 * FS)
ping[:n_fade] *= np.linspace(0, 1, n_fade)
ritardo = 2 * distanza_parete / 343
D = round(ritardo * FS)
y = np.zeros(round(FS * 1.0))
y[:len(ping)] += ping
y[D:D + len(ping)] += peso_riflessione * ping
plt.plot(np.arange(len(y)) / FS, y, color="#1769c2")
plt.xlabel("Tempo [s]"); plt.ylabel("Ampiezza")
plt.title(f"Diretto e riflesso — ritardo {ritardo * 1000:.1f} ms")
plt.tight_layout(); plt.show(); plt.close()
ascolto(y, "Suono diretto e una riflessione")


## Un piccolo esperimento

- Nella demo 1, modificare la frequenza lasciando invariata l’ampiezza.
- Nella demo 2, confrontare 0° e 180° senza cambiare il volume di riproduzione.
- Nella demo 3, passare da 3 a 6 Hz di differenza.
- Nella demo 5, confrontare −6 e +6 dB e osservare il guadagno stampato.
- Nella demo 6, confrontare una parete a 2 m e una a 50 m.

Per ogni prova annotare che cosa cambia nel grafico, che cosa cambia nell’ascolto e quale parametro spiega il risultato.
